In [2]:
import re
import numpy as np
import pandas as pd
import itertools
from tqdm.auto import tqdm

import matplotlib.pyplot as plt


from bert_score import score
import pymorphy3
from rapidfuzz.distance import Levenshtein
from sacrebleu.metrics import BLEU, CHRF


import warnings
import logging
from transformers import logging as hf_logging

warnings.filterwarnings("ignore")

hf_logging.set_verbosity_error()
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("bert_score").setLevel(logging.ERROR)

In [3]:
files = {
    "Финский": "c2_augmented_one_times_translated_fi.csv",
    "Корейский": "c2_augmented_one_times_translated_ko.csv",
    "Зулу": "c2_augmented_one_times_translated_zu.csv",
    "Сербский": "c2_augmented_one_times_translated_sr.csv",
    "Персидский": "c2_augmented_one_times_translated_fa.csv",
    "Турецкий": "c2_augmented_one_times_translated_tr.csv",
    "Арабский": "c2_augmented_one_times_translated_ar.csv",
    "Африкаанс": "c2_augmented_one_times_translated_af.csv",
    "Латинский": "c2_augmented_one_times_translated_la.csv",
    "Греческий": "c2_augmented_one_times_translated_el.csv",
}

# Вспомогательные функции подсчета метрик

In [4]:
morph = pymorphy3.MorphAnalyzer()

def tokenize_words(text):
    text = str(text).lower()
    return re.findall(r"[а-яёa-z]+", text)


def lemmatize_text(text):
    words = tokenize_words(text)
    lemmas = [morph.parse(word)[0].normal_form for word in words]
    return lemmas

In [5]:
def bertscore_pair(original_text, augmented_text, model_type="DeepPavlov/rubert-base-cased"):
    """
    Считает BERTScore между двумя текстами:
    original_text — исходный текст
    augmented_text — сгенерированный / аугментированный текст
    """

    P, R, F1 = score(
        [augmented_text],      # candidate
        [original_text],       # reference
        model_type=model_type,
        num_layers=12,
        lang="ru",
        verbose=False
    )

    return {
        "precision": P.item(),
        "recall": R.item(),
        "f1": F1.item()
    }

In [6]:
bleu_metric = BLEU(effective_order=True)
chrf_metric = CHRF()


def jaccard_similarity_lemmas(original, augmented):
    orig_lemmas = set(lemmatize_text(original))
    aug_lemmas = set(lemmatize_text(augmented))

    if not orig_lemmas and not aug_lemmas:
        return 1.0

    if not orig_lemmas or not aug_lemmas:
        return 0.0

    return len(orig_lemmas & aug_lemmas) / len(orig_lemmas | aug_lemmas)


def common_words_ratio(original, augmented):
    """
    Доля лемм исходного текста, которые сохранились в аугментированном.
    """
    orig_lemmas = set(lemmatize_text(original))
    aug_lemmas = set(lemmatize_text(augmented))

    if not orig_lemmas:
        return 0.0

    return len(orig_lemmas & aug_lemmas) / len(orig_lemmas)


def normalized_levenshtein_distance(original, augmented):
    """
    0 — тексты одинаковые.
    1 — тексты максимально различаются.
    """
    original = str(original)
    augmented = str(augmented)

    max_len = max(len(original), len(augmented))

    if max_len == 0:
        return 0.0

    distance = Levenshtein.distance(original, augmented)
    return distance / max_len


def normalized_levenshtein_similarity(original, augmented):
    """
    1 — тексты одинаковые.
    0 — тексты максимально различаются.
    """
    return 1 - normalized_levenshtein_distance(original, augmented)

In [7]:
def lcs_length(x, y):
    """
    Longest Common Subsequence для двух списков токенов.
    """
    m, n = len(x), len(y)
    dp = [[0] * (n + 1) for _ in range(m + 1)]

    for i in range(m):
        for j in range(n):
            if x[i] == y[j]:
                dp[i + 1][j + 1] = dp[i][j] + 1
            else:
                dp[i + 1][j + 1] = max(dp[i][j + 1], dp[i + 1][j])

    return dp[m][n]


def rouge_l_f1(original, augmented):
    orig_tokens = tokenize_words(original)
    aug_tokens = tokenize_words(augmented)

    if not orig_tokens or not aug_tokens:
        return 0.0

    lcs = lcs_length(orig_tokens, aug_tokens)

    precision = lcs / len(aug_tokens)
    recall = lcs / len(orig_tokens)

    if precision + recall == 0:
        return 0.0

    return 2 * precision * recall / (precision + recall)

In [8]:
def bleu_score(original, augmented):
    """
    BLEU: насколько n-граммы аугментированного текста совпадают с исходным.
    Значение приводим к диапазону 0–1.
    """
    score = bleu_metric.sentence_score(
        hypothesis=str(augmented),
        references=[str(original)]
    ).score

    return score / 100


def chrf_score(original, augmented):
    """
    chrF: сходство на уровне символьных n-грамм.
    Значение приводим к диапазону 0–1.
    """
    score = chrf_metric.sentence_score(
        hypothesis=str(augmented),
        references=[str(original)]
    ).score

    return score / 100

# Сравнение исходных и аугментированных текстов внутри одного языка

In [9]:
def calculate_pair_metrics(original, augmented):
    return {
        "bert_score": bertscore_pair(original, augmented),
        "jaccard_lemmas": jaccard_similarity_lemmas(original, augmented),
        "common_words_ratio": common_words_ratio(original, augmented),
        "levenshtein_distance": normalized_levenshtein_distance(original, augmented),
        "levenshtein_similarity": normalized_levenshtein_similarity(original, augmented),
        "rouge_l": rouge_l_f1(original, augmented),
        "bleu": bleu_score(original, augmented),
        "chrf": chrf_score(original, augmented),
    }

In [10]:
all_rows = []

for lang, path in files.items():
    df = pd.read_csv(path)

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Обработка строк"):
        original = row['text']
        augmented = row['augmented-text']

        metrics = calculate_pair_metrics(original, augmented)

        metrics["language"] = lang
        metrics["original"] = original
        metrics["augmented"] = augmented

        all_rows.append(metrics)

pair_metrics_df = pd.DataFrame(all_rows)

Обработка строк:   0%|          | 0/120 [00:00<?, ?it/s]

Обработка строк:   0%|          | 0/120 [00:00<?, ?it/s]

Обработка строк:   0%|          | 0/120 [00:00<?, ?it/s]

Обработка строк:   0%|          | 0/120 [00:00<?, ?it/s]

Обработка строк:   0%|          | 0/120 [00:00<?, ?it/s]

Обработка строк:   0%|          | 0/120 [00:00<?, ?it/s]

Обработка строк:   0%|          | 0/120 [00:00<?, ?it/s]

Обработка строк:   0%|          | 0/120 [00:00<?, ?it/s]

Обработка строк:   0%|          | 0/120 [00:00<?, ?it/s]

Обработка строк:   0%|          | 0/120 [00:00<?, ?it/s]

In [11]:
pair_metrics_df['bert_score_precision'] = [pair_metrics_df['bert_score'][i]['precision'] for i in range(len(pair_metrics_df))]
pair_metrics_df['bert_score_recall'] = [pair_metrics_df['bert_score'][i]['recall'] for i in range(len(pair_metrics_df))]
pair_metrics_df['bert_score_f1'] = [pair_metrics_df['bert_score'][i]['f1'] for i in range(len(pair_metrics_df))]

del pair_metrics_df['bert_score']

pair_metrics_df

,jaccard_lemmas,common_words_ratio,levenshtein_distance,levenshtein_similarity,rouge_l,bleu,chrf,language,original,augmented,bert_score_precision,bert_score_recall,bert_score_f1
0,0.411765,0.583333,0.430939,0.569061,0.392157,0.196086,0.524398,Финский,"3:38–3:54 Собственно, сама по себе радиация не...",3:38–3:54 На самом деле радиация не заразна. Н...,0.775861,0.759988,0.767842
1,0.689655,0.769231,0.377880,0.622120,0.576923,0.191701,0.631853,Финский,Недавно компания Uber объявила об инвестиции о...,Uber недавно объявил об инвестициях в миллиард...,0.837157,0.811666,0.824215
2,0.681818,0.750000,0.569620,0.430380,0.512821,0.553241,0.689427,Финский,"Множество повестей: «Двойник», «Дядюшкин сон»,...",В творчестве великого писателя было много расс...,0.877201,0.858238,0.867616
3,0.433333,0.541667,0.370370,0.629630,0.510638,0.165823,0.578328,Финский,Встречи одноклассников и одногруппников превра...,Встречи одноклассников и одноклассников превра...,0.711352,0.630465,0.668470
4,0.473684,0.666667,0.347826,0.652174,0.580645,0.354536,0.659025,Финский,Самопрезентация — как главная черта времени и ...,Самопрезентация – это одновременно визитная ка...,0.779142,0.796302,0.787628
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1195,0.750000,0.875000,0.140097,0.859903,0.792453,0.739389,0.848390,Греческий,00:00:24 А сегодня испытания. 13 участников. Ж...,00:00:24 А сегодня испытание. 13 участников. Ж...,0.941863,0.948598,0.945219
1196,0.656250,0.777778,0.413462,0.586538,0.607143,0.274395,0.634205,Греческий,«Прома» способна предложить дилерам вполне бож...,«Прома» способна предложить дилерам очень выго...,0.843513,0.803180,0.822853
1197,0.920000,0.958333,0.074074,0.925926,0.938776,0.899648,0.962367,Греческий,Tesla — не единственный игрок на рынке: больши...,Tesla — не единственный игрок на рынке: больши...,0.977269,0.976563,0.976916
1198,0.636364,0.807692,0.465241,0.534759,0.592593,0.246420,0.566381,Греческий,"Понятно, что среди крупнейших стран – поставщи...","Понятно, что в число крупнейших стран, поставл...",0.806652,0.804121,0.805384


In [12]:
metric_columns = [
    "bert_score_precision",
    "bert_score_recall",
    "bert_score_f1",
    "jaccard_lemmas",
    "common_words_ratio",
    "levenshtein_distance",
    "levenshtein_similarity",
    "rouge_l",
    "bleu",
    "chrf",
]

results_df = (
    pair_metrics_df
    .groupby("language")[metric_columns]
    .agg(["mean", "std", "min", "max"])
)

In [13]:
results_df_flat = results_df.copy()

results_df_flat.columns = [
    f"{metric}_{stat}"
    for metric, stat in results_df_flat.columns
]

results_df_flat = results_df_flat.reset_index()

In [14]:
results_df['bert_score_precision']

,mean,std,min,max
language,,,,
Арабский,0.826666,0.083747,0.546730,1.000000
Африкаанс,0.858830,0.078429,0.576304,1.000000
Греческий,0.849201,0.076319,0.603098,0.993837
Зулу,0.806030,0.079921,0.545444,0.974463
Корейский,0.778388,0.089965,0.522048,0.971402
Латинский,0.785433,0.092536,0.544622,0.981116
Персидский,0.814894,0.081465,0.539353,1.000000
Сербский,0.851726,0.075226,0.645697,1.000000
Турецкий,0.821680,0.082461,0.614730,0.988303


In [15]:
results_df['bert_score_recall']

,mean,std,min,max
language,,,,
Арабский,0.814495,0.088497,0.579536,1.000000
Африкаанс,0.844454,0.082835,0.628938,1.000000
Греческий,0.836842,0.081042,0.640006,0.989982
Зулу,0.788366,0.087513,0.524820,0.974021
Корейский,0.764504,0.096893,0.491815,0.984166
Латинский,0.764485,0.098553,0.543470,0.981116
Персидский,0.795491,0.089678,0.586666,1.000000
Сербский,0.838253,0.082919,0.639972,1.000000
Турецкий,0.810141,0.089795,0.586063,0.994562


In [16]:
results_df['bert_score_f1']

,mean,std,min,max
language,,,,
Арабский,0.820269,0.085117,0.572894,1.000000
Африкаанс,0.851383,0.079823,0.601472,1.000000
Греческий,0.842734,0.077669,0.633513,0.990496
Зулу,0.796775,0.082630,0.540757,0.974242
Корейский,0.771083,0.092504,0.506481,0.977742
Латинский,0.774497,0.094523,0.544045,0.981116
Персидский,0.804730,0.084462,0.564757,1.000000
Сербский,0.844712,0.078287,0.649098,1.000000
Турецкий,0.815657,0.085415,0.601817,0.991423


In [17]:
results_df['jaccard_lemmas']

,mean,std,min,max
language,,,,
Арабский,0.599803,0.165461,0.187500,1.000000
Африкаанс,0.653057,0.154982,0.187500,1.000000
Греческий,0.630337,0.152084,0.258065,1.000000
Зулу,0.547052,0.144796,0.151515,0.920000
Корейский,0.525945,0.141370,0.225000,0.916667
Латинский,0.525404,0.157433,0.147059,0.920000
Персидский,0.575105,0.146912,0.156250,1.000000
Сербский,0.651960,0.148224,0.187500,1.000000
Турецкий,0.585511,0.152387,0.225806,1.000000


In [18]:
results_df['common_words_ratio']

,mean,std,min,max
language,,,,
Арабский,0.733525,0.135488,0.333333,1.000000
Африкаанс,0.770133,0.123311,0.333333,1.000000
Греческий,0.756446,0.121348,0.444444,1.000000
Зулу,0.687819,0.130311,0.277778,0.958333
Корейский,0.670832,0.128460,0.375000,0.956522
Латинский,0.660820,0.140916,0.277778,0.958333
Персидский,0.709385,0.130195,0.277778,1.000000
Сербский,0.771617,0.118533,0.333333,1.000000
Турецкий,0.719815,0.130891,0.350000,1.000000


In [19]:
results_df['levenshtein_distance']

,mean,std,min,max
language,,,,
Арабский,0.308227,0.127315,0.000000,0.569620
Африкаанс,0.257964,0.125310,0.000000,0.581731
Греческий,0.268763,0.113833,0.020619,0.465241
Зулу,0.338291,0.116909,0.062112,0.578947
Корейский,0.405274,0.147878,0.030675,0.764706
Латинский,0.371831,0.134679,0.033654,0.794872
Персидский,0.334399,0.134842,0.000000,0.725581
Сербский,0.262244,0.124387,0.000000,0.666667
Турецкий,0.344575,0.144588,0.019231,0.780702


In [20]:
results_df['levenshtein_similarity']

,mean,std,min,max
language,,,,
Арабский,0.691773,0.127315,0.430380,1.000000
Африкаанс,0.742036,0.125310,0.418269,1.000000
Греческий,0.731237,0.113833,0.534759,0.979381
Зулу,0.661709,0.116909,0.421053,0.937888
Корейский,0.594726,0.147878,0.235294,0.969325
Латинский,0.628169,0.134679,0.205128,0.966346
Персидский,0.665601,0.134842,0.274419,1.000000
Сербский,0.737756,0.124387,0.333333,1.000000
Турецкий,0.655425,0.144588,0.219298,0.980769


In [21]:
results_df['rouge_l']

,mean,std,min,max
language,,,,
Арабский,0.635310,0.152244,0.222222,1.000000
Африкаанс,0.686922,0.147193,0.250000,1.000000
Греческий,0.671539,0.138308,0.277778,0.980392
Зулу,0.597038,0.138871,0.228571,0.897959
Корейский,0.527208,0.160485,0.171429,0.958333
Латинский,0.560674,0.162882,0.057143,0.958333
Персидский,0.610229,0.147387,0.256410,1.000000
Сербский,0.680308,0.143511,0.250000,1.000000
Турецкий,0.607182,0.150391,0.193548,1.000000


In [22]:
results_df['bleu']

,mean,std,min,max
language,,,,
Арабский,0.387994,0.192924,0.046214,1.000000
Африкаанс,0.457187,0.206963,0.068157,1.000000
Греческий,0.432305,0.191275,0.060012,0.910769
Зулу,0.346897,0.172420,0.065334,0.871393
Корейский,0.284630,0.180499,0.046793,0.898540
Латинский,0.310076,0.184774,0.020408,0.925752
Персидский,0.357725,0.188575,0.058001,1.000000
Сербский,0.452220,0.200754,0.097257,1.000000
Турецкий,0.359403,0.184727,0.046044,0.902514


In [23]:
results_df['chrf']

,mean,std,min,max
language,,,,
Арабский,0.657978,0.128058,0.364843,1.000000
Африкаанс,0.707820,0.123309,0.406222,1.000000
Греческий,0.687786,0.119747,0.428193,0.970355
Зулу,0.610885,0.120245,0.344970,0.948998
Корейский,0.591089,0.120306,0.342171,0.950266
Латинский,0.586768,0.133537,0.296808,0.943979
Персидский,0.630407,0.125448,0.355862,1.000000
Сербский,0.701150,0.122084,0.406599,1.000000
Турецкий,0.646910,0.126124,0.300634,0.977062
